# Upload to S3

Upload the processed `.txt` files (from `2_clean_data.ipynb`) to the S3 prefix that feeds the matching Bedrock Knowledge Base:

- `sources_txt/fixed_size/<source>/` feeds the fixed-chunking KB (`FIXED_KNOWLEDGE_BASE_ID`) — chunked at 500 tokens with 10% overlap
- `sources_txt/no_chunking/<source>/` feeds the no-chunking KB (`NOCHUNK_KNOWLEDGE_BASE_ID`)

In [ ]:
import os

import boto3


bucket = "enterprise-rag-bench-yuk-us-east-1-replica"

sources = ["confluence", "fireflies", "github", "gmail", "google_drive", "hubspot", "jira", "linear", "slack"]

# Which KB prefix(es) each source's processed text feeds
source_kb_map = {"confluence": ["fixed_size"],
                "fireflies": ["fixed_size"],
                "github": ["fixed_size"],
                "gmail": ["no_chunking"],
                "google_drive": ["fixed_size"],
                "hubspot": ["no_chunking"],
                "jira": ["no_chunking"],
                "linear": ["no_chunking"],
                "slack": ["no_chunking"],
                }


In [ ]:
# Upload every file under local_folder to s3://bucket/s3_prefix, preserving structure.
def upload_local_folder(local_folder, s3_prefix):

    s3 = boto3.client("s3")

    count = 0

    for root, _, files in os.walk(local_folder):

        for file in files:

            local_path = os.path.join(root, file)

            relative_path = os.path.relpath(local_path, local_folder)
            s3_key = os.path.join(s3_prefix, relative_path).replace("\\", "/")

            print(f"Uploading {local_path} -> s3://{bucket}/{s3_key}")

            s3.upload_file(local_path, bucket, s3_key)

            count += 1

    print(f"Finished! Uploaded {count} files from {local_folder}.")


# Upload a source's processed_sources/<source>/ folder to every KB prefix it feeds.
def upload_source(source):

    local_folder = f"processed_sources/{source}"

    for prefix_type in source_kb_map[source]:
        s3_prefix = f"sources_txt/{prefix_type}/{source}"
        upload_local_folder(local_folder, s3_prefix)


for source in sources:
    upload_source(source)
